In [1]:
import pandas as pd
import numpy as np
import os
from openai import OpenAI

In [2]:
# change to each experiment ID
exp_id = "10-spanish bless prompt"

In [3]:
from dotenv import load_dotenv
load_dotenv(os.path.expanduser("~/final_project_openrouter/.env"))

True

In [4]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

In [5]:
#Loading in CLEARS dataset

#150 words roughly equate to 200 tokens from https://platform.openai.com/tokenizer

def load_clears_paired(csv_path, max_orig_words=150):
    df = pd.read_csv(csv_path)
    txt = df[df['type'] == 'TXT'][['row_id', 'text']].rename(columns={'text': 'original'})
    fac = df[df['type'] == 'FAC'][['row_id', 'text']].rename(columns={'text': 'simplified'})
    paired = txt.merge(fac, on='row_id')

    #filter texts that fit within token limit of 512 + with prompt 
    paired['orig_words'] = paired['original'].str.split().str.len()
    paired = paired[paired['orig_words'] <= max_orig_words].drop(columns='orig_words')
    return paired.reset_index(drop=True)

# Use the provided train/test splits
data_train = load_clears_paired('/home/c23068554/final_project/datasets/cleartext-discriminativo-es/train.csv')
data_test = load_clears_paired('/home/c23068554/final_project/datasets/cleartext-discriminativo-es/test.csv')

print(f"Train pairs (filtered): {len(data_train)}")
print(f"Test pairs (filtered): {len(data_test)}")

Train pairs (filtered): 427
Test pairs (filtered): 173


In [6]:
data_test = data_test.head(10)
print(f"TEST MODE: running on {len(data_test)} paragraphs only")

TEST MODE: running on 10 paragraphs only


In [7]:
random_state = 59
examples = data_train.sample(n=3, random_state = random_state)

## Prompt and Few-shot construction



In [8]:
instruction = "Por favor, reformula la siguiente oración compleja para que sea más comprensible para hablantes no nativos de español. Puedes hacerlo reemplazando palabras complejas por sinónimos más sencillos (parafraseando), eliminando información irrelevante (condensando) o dividiendo la oración en varias más simples. La oración simplificada final debe ser gramaticalmente correcta, fluida y conservar las ideas principales de la original sin alterar su significado.\n\n"
def makePrompt(instruction, examples):
  #formatting text for fewshot examples
  fewshot = ""
  for index, row in examples.iterrows():
    fewshot += (f"Compleja: {row.loc['original']}\nSimplificada: {row.loc['simplified']}\n\n")
  return(instruction + fewshot)

fewshot_example = makePrompt(instruction, examples)
print(fewshot_example)

Por favor, reformula la siguiente oración compleja para que sea más comprensible para hablantes no nativos de español. Puedes hacerlo reemplazando palabras complejas por sinónimos más sencillos (parafraseando), eliminando información irrelevante (condensando) o dividiendo la oración en varias más simples. La oración simplificada final debe ser gramaticalmente correcta, fluida y conservar las ideas principales de la original sin alterar su significado.

Compleja: Los días 17 y 18 de marzo por motivo técnicos el Museo de Hogueras permanecerá cerrado, volviendo a reabrir sus puertas a partir del día 20 de marzo.
Simplificada:El 17 y 18 de marzo cierra el Museo de Hogueras
 El 17 y 18 de marzo el Museo de Hogueras cerrará para hacer unos arreglos.
 El 20 de marzo el Museo de Hogueras vuelve a estar abierto.

Compleja: Antonio Manresa, ha acudido hoy a la inauguración de la exposición XXI Premios de Fotografía de Fogueres "Memorial Reme Vélez", que acoge la renovada sala Ámbito Cultural de 

## Load in model and inference

### Decoder models

In [9]:
# Decoder models for Spanish 
mistral_7b = "mistralai/mistral-7b-instruct-v0.1"
llama31_8b = "meta-llama/llama-3.1-8b-instruct"
gemma2_9b = "google/gemma-2-9b-it"
qwen2_7b = "qwen/qwen-2.5-7b-instruct"
llama33_70b = "meta-llama/llama-3.3-70b-instruct"

In [10]:
import time
model_id = mistral_7b

def generate(prompt, max_tokens=500):
    for attempt in range(5):
        try:
            response = client.chat.completions.create(
                model=model_id,
                messages=[
                    {"role": "system", "content": " Responda únicamente con el texto simplificado."},
                    {"role": "user", "content": prompt}],
                max_tokens=max_tokens,
                temperature=1.0,
                top_p=0.9,
            )
            content = response.choices[0].message.content
            if content is None:
                print(f"  Empty response (attempt {attempt+1})")
                time.sleep(1)
                continue
            return content.strip()
        except Exception as e:
            if "rate" in str(e).lower() or "429" in str(e):
                wait = 2 ** attempt
                print(f"  Rate limited, retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("Max retries exceeded")

## Looping through dataset with prompt

In [11]:
#looping through all n-shot prompt
def promptLoop(fewshot_example, data_test):
  LMsimplified = []
  for i, row in enumerate(data_test['original']):
    full_prompt = fewshot_example + f"Compleja: {row}\nSimplficada:"
    LMsimplified.append(generate(full_prompt))
    #progress check on infference
    if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(data_test)} done")
  return LMsimplified

In [12]:
print(fewshot_example + f"Complex:\nSimple:")

Por favor, reformula la siguiente oración compleja para que sea más comprensible para hablantes no nativos de español. Puedes hacerlo reemplazando palabras complejas por sinónimos más sencillos (parafraseando), eliminando información irrelevante (condensando) o dividiendo la oración en varias más simples. La oración simplificada final debe ser gramaticalmente correcta, fluida y conservar las ideas principales de la original sin alterar su significado.

Compleja: Los días 17 y 18 de marzo por motivo técnicos el Museo de Hogueras permanecerá cerrado, volviendo a reabrir sus puertas a partir del día 20 de marzo.
Simplificada:El 17 y 18 de marzo cierra el Museo de Hogueras
 El 17 y 18 de marzo el Museo de Hogueras cerrará para hacer unos arreglos.
 El 20 de marzo el Museo de Hogueras vuelve a estar abierto.

Compleja: Antonio Manresa, ha acudido hoy a la inauguración de la exposición XXI Premios de Fotografía de Fogueres "Memorial Reme Vélez", que acoge la renovada sala Ámbito Cultural de 

### Inference

In [13]:
LMoutput = (promptLoop(fewshot_example, data_test))
#print(promptLoop(fewshot_example, data_test))

In [14]:
# Clean prompt leakage from decoder outputs, using common LLM speech 
def clean_output(text):
    
    text = text.lstrip(':.,;>•*- \n')

    for delim in ['\nComplex:', '\n\nComplex:', '\nNote:', '\n(Note:', '\nAnd also', '\ncomplex sentence']:
        if delim in text:
            text = text.split(delim)[0]
    return text.strip()

LMoutput = [clean_output(s) for s in LMoutput]

### Organising outputs

In [15]:
reference = data_test['simplified'].tolist()
source = data_test['original'].tolist()

# CSV WRITE

In [16]:
import csv

out_path = "lm_output.csv"

with open(out_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(["Exp ID", "Index", "Original", "Reference", "LM Output"])
    for i, (orig, ref, lm) in enumerate(zip(source, reference, LMoutput)):
        writer.writerow([exp_id, i, orig, ref, lm])

print(f"Wrote {len(source)} rows to {out_path}")

Wrote 10 rows to lm_output.csv


# Evaluation

### Copy sentences

In [17]:
count = 0
for original, simple, lmSimple in zip(data_test['original'], data_test['simplified'], LMoutput):
  print(original)
  print(simple)
  print(lmSimple + "\n")
  if original == lmSimple:
    #print(original)
    #print(simple)
    #print(lmSimple)
    count += 1
print(f"Number of sentences not simplified or altered by the model: {count}")

La nueva junta directiva de la Societat Musical ‘La Nova’ de Benidorm ha presentado esta tarde sus credenciales en el Ayuntamiento de Benidorm, en donde se ha entrevistado con el alcalde, Toni Pérez, y el concejal de Cultura, Jaime Jesús Pérez.


El presidente, Juan Vicente Moreno, el director de la banda, Francisco José Rovira Peretó y otros miembros de la entidad, han mantenido una reunión en la que han expuesto al primer edil su programa de actividades previsto para los próximos meses de una marcada línea de continuidad con los ejercicios anteriores.
La nueva directiva de la Sociedad Musical La Nova de Benidorm visita el ayuntamiento 
La nueva junta directiva de la Sociedad Musical La Nova de Benidorm 
presentó esta tarde sus credenciales en el Ayuntamiento de Benidorm, 
con el alcalde y el concejal de Cultura.
El presidente, el director de la banda y otros miembros de la entidad, 
estuvieron en la reunión con el alcalde presentando 
el programa de actividades para los próximos mese

In [18]:
import evaluate

#load metrics
rouge = evaluate.load('rouge')
bleu = evaluate.load('bleu')
bertscore = evaluate.load('bertscore')
sari = evaluate.load('sari')

#Converting simplified reference sentences into a list inside a list
sari_references = [[s] for s in data_test["simplified"].astype(str).tolist()]
sari_score = sari.compute(sources= source, predictions= LMoutput, references= sari_references)

# Compute scores
rouge_results = rouge.compute(predictions= LMoutput, references = reference)
bleu_results = bleu.compute(predictions= LMoutput, references = reference)

bertscore_compute = bertscore.compute(predictions=LMoutput, references= reference, lang='en')
berstcoreAvg = np.mean(bertscore_compute['f1'])

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Fernandex Huerta Score

In [19]:
#fernandex_huerta score
import textstat
textstat.set_lang("es")

fhOrig = np.mean([textstat.fernandez_huerta(s) for s in data_test["original"].tolist()])
fhSimp = np.mean([textstat.fernandez_huerta(s) for s in LMoutput])

In [20]:
#output
print(f"Dataset: CLEARS (FAC), size: {len(data_test)}, random state: {random_state}")
print(f"Model: {model_id}")
print()
print(f"ROUGE Score: {rouge_results['rouge1']}")
print(f"BLEU Score: {bleu_results['bleu']}")
print(f"BERTScore Score (xlm-roberta-large): {berstcoreAvg}")
print(f"SARI Score: {sari_score['sari']}")
print()
print(f"Original Gulpease Index: {fhOrig}")
print(f"Simplified Gulpease Index: {fhSimp}")
print()
print(f"unsimplified sentences: {count}/{len(data_test)}")

Dataset: CLEARS (FAC), size: 10, random state: 59
Model: mistralai/mistral-7b-instruct-v0.1

ROUGE Score: 0.30557440665706137
BLEU Score: 0.0954966716033527
BERTScore Score (xlm-roberta-large): 0.8460442006587983
SARI Score: 37.52395758527107

Original Gulpease Index: 58.536293900478356
Simplified Gulpease Index: 68.38661619403248

unsimplified sentences: 0/10


In [21]:
print(f"\nTAB-SEPARATED (paste into experiment sheet metrics columns):")
print(f"{sari_score['sari']:.4f}\t{bleu_results['bleu']:.4f}\t{berstcoreAvg:.4f}\t{fhSimp:.4f}\t{count}/{len(data_test)}")


TAB-SEPARATED (paste into experiment sheet metrics columns):
37.5240	0.0955	0.8460	68.3866	0/10


In [22]:
# command to clear cache often, to reduce disk space used:
# rm -rf ~/.cache/*

#check disk space used:
# du -h --max-depth=1 ~ | sort -h

'''
Exporting CSV:
On local:
scp -r c23068554@10.98.84.2:/home/c23068554/final_project/lm_output.csv '/Users/justinwoodham/Desktop/CS/Y3/Final Year Project/raw outputs'
On Remote: 
rm lm_output.csv

'''

# remove .env using: rm ~/final_project_openrouter/.env

"\nExporting CSV:\nOn local:\nscp -r c23068554@10.98.84.2:/home/c23068554/final_project/lm_output.csv '/Users/justinwoodham/Desktop/CS/Y3/Final Year Project/raw outputs'\nOn Remote: \nrm lm_output.csv\n\n"